# Programming Assignment 3: Text Representation - Distributed Method
**Twitter Entity Sentiment - Word2Vec, GloVe, FastText and document vectors**

## 📋 Assignment Information

**Course:** Unstructured Data Analysis (2026-2)
**Assignment:** Programming Assignment 3 - Text Representation II (Distributed Method)
**Released:** October 12, 2026 (Mon), 09:00
**Deadline:** October 19, 2026 (Mon), 23:59
**Total:** 100 points

## 🎯 Objectives

Assignment 2 represented each tweet as a sparse count / TF-IDF vector. This assignment replaces it with the
**dense, learned** representations of lecture 5-1 / 5-2, and makes you open each model up:

- Generate the **skip-gram training pairs** and compute the **subsampling** probability of Word2Vec
- Use **both** vectors of a word, $u$ and $v$, in the skip-gram softmax $p(o \mid c)$
- Build the GloVe **co-occurrence matrix** and its **probability ratios**, and the weighting function $f(x)$
- Solve word analogies **by hand** and find out what `most_similar` does that the formula does not
- Rebuild a FastText vector from its **character n-grams**
- Build document vectors by **averaging** word vectors and with **Doc2Vec** (PV-DM / PV-DBOW)

Everything is inside lecture 5-1 / 5-2 and the Week 6 notebooks. No classifier is trained, and no t-SNE (Week 7).

The data and settings differ from the Week 6 notebooks on purpose (tweets instead of movie reviews, 50 dimensions,
window 2): the methods are the same, the numbers are not, so do not compare your outputs with the class notebooks.

**Your assignment is personal.** Section 0 asks for your student ID and derives from it the words, word pairs,
analogy sections and tweets you work on (`MY`). Your hidden numbers are therefore different from your classmates'.

## 📝 Instructions

1. `File ▸ Save a copy in Drive`, then rename the notebook to **`Assignment_3_YourName_StudentID.ipynb`** (e.g. `Assignment_3_HongGilDong_2026123456.ipynb`).
2. Fill in every `________` blank. A blank may be a value, an expression or a whole right-hand side.
   **Do not** change the code outside the blanks and **do not** rename variables: the grader re-runs your notebook.
3. `Runtime ▸ Restart and run all` must finish without an error. **Keep the outputs** in the notebook when you submit.
4. Submit the `.ipynb` file to **"Programming Assignment 3 – Text Representation II"** under *Assignments*.

**There are no free-text discussion questions.** 80 points are for code: the grader re-runs your notebook and compares
**its** outputs with the reference values computed for **your** student ID. The last 20 points are five short questions
(Part 5), each with exactly one correct answer: one value read from **your own** outputs and one multiple-choice item.

> ⚠️ **Read each specification to the letter.** Several TODOs deliberately ask for something that is *not* the default
> behavior of a library, nor the most common version you will find online or get from a chatbot
> (for example: which matrix a vector comes from, what is counted in a denominator, or whether input words are excluded).
> Only some expected outputs are printed below each TODO, as **checks**; the rest are known only to the grader.
> A blank that "looks right" but ignores the specification produces different numbers and loses its points.

| Part | Content | Points |
|---|---|---|
| 1 | Word2Vec: training pairs, subsampling, the two vectors (TODO 1–3) | 24 |
| 2 | GloVe: co-occurrence, weighting, analogies (TODO 4–6) | 24 |
| 3 | FastText: character n-grams, composing a vector (TODO 7–8) | 12 |
| 4 | Document level: averaging, Doc2Vec (TODO 9–10) | 20 |
| 5 | Reading your own results: Q1–Q5 | 20 |
| | **Total** | **100** |

Partial credit is given per blank (see the points next to each TODO).

## 📂 Dataset

The **same** Twitter Entity Sentiment file as Assignments 1 and 2. Section 0 repeats the Assignment 1 cleaning and
gives you **two** token lists per tweet:

| variable | content | used by |
|---|---|---|
| `tweets_full` | tokens **with** stop words | Word2Vec, subsampling, FastText (they need the full context) |
| `tweets` | tokens **without** stop words (`tokens_clean` of Assignment 2) | GloVe counts, document vectors |

and the pre-trained **GloVe** vectors `glove` (Wikipedia + Gigaword, 6B tokens, **50** dimensions; ~66 MB, about one minute to download).

## &nbsp;0. Setup (given)
*No TODO in this section* except your student ID. Run it once and check the printed numbers:
`documents: 10282`, `tokens (full): 171339`, `tokens (clean): 101060`, `GloVe: 400000 x 50`.

> If Colab asks you to **restart the session** after the install, do it and run this section again.

In [ ]:
!pip install -q gensim

import nltk
for r in ['punkt', 'punkt_tab', 'stopwords']:
    nltk.download(r, quiet=True)

In [ ]:
import re
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
from sklearn.metrics.pairwise import cosine_similarity

import gensim
import gensim.downloader as api
from gensim.models import Word2Vec, FastText
from gensim.models.fasttext import ft_ngram_hashes
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.width', 140)
print('gensim', gensim.__version__)

In [ ]:
# --- the Assignment 1 pipeline, given ---------------------------------------
DATA_URL = "https://raw.githubusercontent.com/snhzyn/2026-2-Unstructured-Data-Analysis/main/week03-text-preprocessing/data/twcs.csv"

data = pd.read_csv(DATA_URL)
data_unique = data.drop_duplicates(subset='idx', keep='first')
df = data_unique[data_unique['Sentiment'] != 'Irrelevant'].reset_index(drop=True)


def clean_text(d):
    d = str(d).lower()
    d = re.sub(r"http\S+|www\S+|\S+\.com\S*", " ", d)
    d = re.sub(r"[^a-z'\s]", " ", d)
    return d


CUSTOM_STOPS = ['im', 'ive', 'dont', 'cant', 'thats', 'like', 'get', 'got', 'one',
                'game', 'games', 'amp']
stop_words = set(stopwords.words('english')) | set(CUSTOM_STOPS)
tokenizer = RegexpTokenizer(r"[a-z']{2,}")

df['tokens'] = df['Review'].apply(clean_text).apply(tokenizer.tokenize)
df['tokens_clean'] = df['tokens'].apply(lambda ts: [t for t in ts if t not in stop_words])

tweets_full = df['tokens'].tolist()          # with stop words
tweets = df['tokens_clean'].tolist()         # without stop words

print('documents     :', len(df))
print('tokens (full) :', sum(len(t) for t in tweets_full))
print('tokens (clean):', sum(len(t) for t in tweets))

In [ ]:
glove = api.load('glove-wiki-gigaword-50')       # about one minute
k2i = glove.key_to_index                          # word -> row index
print('GloVe:', glove.vectors.shape[0], 'x', glove.vector_size)

### Your student ID
Type **your own** 10-digit student ID as a string. Everything personal in this assignment is derived from it.
A wrong ID gives you someone else's questions: your hidden values then match nobody, and Part 5 cannot be graded.

In [ ]:
STUDENT_ID = '2026000000'      # <- replace with YOUR student ID

assert STUDENT_ID.isdigit() and len(STUDENT_ID) == 10 and STUDENT_ID != '2026000000', 'Type your own 10-digit student ID'

In [ ]:
# --- your personal parameters (given, do not edit) ---------------------------
# Every student gets different words, pairs, sections and queries, derived from
# STUDENT_ID. The grader recomputes them from your ID, so the ID must be yours.
import hashlib


def _pick(options, salt, k=1):
    order = sorted(range(len(options)),
                   key=lambda n: hashlib.sha256(f'{STUDENT_ID.strip()}|{salt}|{n}'.encode()).hexdigest())
    return [options[n] for n in order[:k]]


_FREQ = ['borderlands', 'love', 'fix', 'play', 'new', 'ps', 'nvidia', 'amazon', 'wait', 'johnson', 'verizon',
         'google', 'facebook', 'playing', 'good', 'twitch', 'overwatch', 'fifa', 'nba', 'dota', 'creed']
_RARE = ['unplayable', 'fortnite', 'server', 'servers', 'update', 'graphics', 'console', 'controller', 'patch',
         'stream', 'madden', 'cyberpunk', 'assassins', 'hearthstone']
_CENTERS = ['fix', 'play', 'update', 'amazon', 'google', 'nvidia', 'borderlands', 'verizon', 'twitch', 'fifa',
            'nba', 'overwatch', 'patch']
_PAIRS = [('amazon', 'google', ['video', 'find', 'via', 'interesting']),
          ('fifa', 'nba', ['best', 'servers', 'give', 'points']),
          ('xbox', 'playstation', ['series', 'love', 'play', 'pc']),
          ('verizon', 'facebook', ['phone', 'twitter', 'live', 'instagram']),
          ('fortnite', 'overwatch', ['season', 'tv', 'good', 'played']),
          ('google', 'facebook', ['play', 'instagram', 'check', 'account']),
          ('amazon', 'microsoft', ['video', 'store', 'today', 'rewards']),
          ('hearthstone', 'overwatch', ['excited', 'fun', 'play', 'gonna']),
          ('dota', 'league', ['esports', 'legends', 'love', 'playing']),
          ('xbox', 'ps', ['microsoft', 'months', 'controller', 'series'])]
_SECTIONS = ['capital-common-countries', 'capital-world', 'currency', 'city-in-state', 'family',
             'gram1-adjective-to-adverb', 'gram2-opposite', 'gram3-comparative', 'gram4-superlative',
             'gram5-present-participle', 'gram6-nationality-adjective', 'gram7-past-tense', 'gram8-plural',
             'gram9-plural-verbs']
_NGRAM_WORDS = ['where', 'unplayable', 'graphics', 'console', 'servers', 'johnson', 'borderlands', 'update',
                'fortnite', 'playstation', 'controller', 'overwatch']
_OOV = ['xboxseriesx', 'playstation5', 'unplayabel', 'fortnitee', 'nvidiartx', 'borderlandz', 'gamepasss',
        'overwatchh', 'hearthstoned', 'johnsonandjohnson']
_INV = ['unplayable', 'borderlands', 'graphics', 'servers', 'update', 'console', 'controller', 'fortnite',
        'overwatch', 'playstation', 'cyberpunk', 'hearthstone']
_QUERIES = [5320, 8476, 9275, 4506, 6704, 1078, 5841, 385, 5613, 1856, 2461, 2565, 200, 300, 8093,
            5188, 8818, 8704, 7352, 4086, 6729, 614, 7301, 10114, 897, 6312, 4630, 7506, 3049, 4537,
            10228, 6009]
_NEW_TWEETS = [['xbox', 'series', 'price', 'revealed'], ['servers', 'down', 'again', 'fix'],
               ['new', 'update', 'patch', 'notes'], ['amazon', 'prime', 'gift', 'card'],
               ['fifa', 'ultimate', 'team', 'packs'], ['nvidia', 'rtx', 'cards', 'sold', 'out'],
               ['borderlands', 'dlc', 'release', 'date'], ['twitch', 'stream', 'live', 'tonight']]

MY = {
    'words': _pick(_FREQ, 'freq', 2) + _pick(_RARE, 'rare', 1),      # TODO 2
    'center': _pick(_CENTERS, 'center')[0],                          # TODO 3
    'pair': _pick(_PAIRS, 'pair')[0],                                # TODO 4
    'sections': _pick(_SECTIONS, 'sections', 4),                     # TODO 6
    'ngram_words': _pick(_NGRAM_WORDS, 'ngram', 2),                  # TODO 7
    'oov': _pick(_OOV, 'oov')[0],                                    # TODO 8 (a)
    'inv': _pick(_INV, 'inv')[0],                                    # TODO 8 (b)
    'query': _pick(_QUERIES, 'query')[0],                            # TODO 9
    'new_tweet': _pick(_NEW_TWEETS, 'tweet')[0],                     # TODO 10
}


for k, v in MY.items():
    print(f'{k:<12} {v}')

## &nbsp;1. Word2Vec (24 pts)

### TODO 1: Skip-gram training pairs (6 points)

Write `skipgram_pairs(tokens, m)` exactly as on the lecture slide *Learning strategy*: for every position `i`, one
`(center, context)` pair for **every** word within `m` positions to the left and right, from left to right.

* The window is **fixed**. (gensim actually shrinks the window at random for every center word, which is *not* what we want here.)
* Pairs never cross the boundary between two tweets: count the pairs **tweet by tweet** and add them up.

**Check:** the test tweet gives
`[('xbox', 'series'), ('xbox', 'x'), ('series', 'xbox'), ('series', 'x'), ('series', 'specs')]` and `#pairs in the test tweet: 18`.
The corpus total is not shown.

In [ ]:
# TODO 1 (6 pts): fixed-window skip-gram pairs

def skipgram_pairs(tokens, m):
    pairs = []
    for i, center in enumerate(tokens):
        for j in range(________, ________):
            if ________:
                pairs.append(________)
    return pairs


test = 'xbox series x specs revealed today'.split()
print(skipgram_pairs(test, 2)[:5])
print('#pairs in the test tweet:', len(skipgram_pairs(test, 2)))

n_pairs = ________
print('#training pairs in the corpus (m=2):', n_pairs)

### TODO 2: Subsampling (8 points)

Use the lecture formula. $P(w_i)$ is the probability that one occurrence of $w_i$ is **removed**:

$$P(w_i) = 1 - \sqrt{\frac{t}{f(w_i)}}$$

* $f(w_i)$ is the **relative** frequency: the count of $w_i$ divided by the total number of tokens of `tweets_full`.
* A word with $f(w_i) < t$ is never removed: clip negative values to **0**.
* Do **not** use gensim's internal formula (it computes a probability of *keeping* with a different expression).

Then report how many tokens survive *in expectation*, $\sum_w \text{count}(w) \times (1 - P(w))$, for the slide's
threshold $t = 10^{-5}$ and for gensim's default $t = 10^{-3}$.

The first line is the check word `the`; the next three are **your** words `MY['words']`.

**Check:** `the  t=1e-05: 0.9828`. The other numbers are not shown.

In [ ]:
# TODO 2 (8 pts): the lecture's subsampling probability

counts = Counter(w for t in tweets_full for w in t)
total = ________


def discard_prob(word, t):
    f = ________
    return ________


for w in ['the'] + MY['words']:
    print(f'{w:<12} t=1e-05: {discard_prob(w, 1e-5):.4f}   t=1e-03: {discard_prob(w, 1e-3):.4f}')

for t in [1e-5, 1e-3]:
    kept = ________
    print(f't={t:g}: expected #tokens kept = {kept:,.0f} of {total:,} ({kept / total:.1%})')

### TODO 3: The two vectors of a word (10 points)

The model below is given (`workers=1` and `seed=42` make it reproducible; it trains in a few seconds).
Lecture slide: every word has **two** vectors, the center vector $v$ (a row of $W$, gensim `w2v.wv`) and the
outside vector $u$ (a column of $W'$, gensim stores it with negative sampling in `w2v.syn1neg`).

Implement the skip-gram softmax over the **whole vocabulary**

$$p(o \mid c) = \frac{\exp(u_o^\top v_c)}{\sum_{w=1}^{W} \exp(u_w^\top v_c)}$$

and print the 5 most probable **context** words of `xbox` (the check) and of **your** word `MY['center']`, next to the 5 most **similar** words.
(As in Notebook 1 §2-5, the model was trained with negative sampling, so this full softmax is an illustration of the
lecture's formula computed from the trained vectors.)

**Check:** both distributions sum to `1.0`, and the most probable context word of `xbox` is `series`.

In [ ]:
w2v = Word2Vec(sentences=tweets_full, vector_size=50, window=2, min_count=3,
               sg=1, negative=5, workers=1, seed=42, epochs=10)
print('vocabulary:', len(w2v.wv), ' W:', w2v.wv.vectors.shape, " W':", w2v.syn1neg.shape)

In [ ]:
# TODO 3 (10 pts): p(o | c) with the outside vectors u and the center vector v

def context_distribution(center):
    v_c = ________
    scores = ________              # u_w . v_c for every word w of the vocabulary
    p = np.exp(scores - scores.max())         # (subtracting the max only avoids overflow)
    return ________


for c in ['xbox', MY['center']]:
    p = context_distribution(c)
    top = ________
    print(f'{c:<12} sum={p.sum():.3f}  context :', [w2v.wv.index_to_key[i] for i in top])
    print('                         similar :', [w for w, _ in w2v.wv.most_similar(c, topn=5)])

## &nbsp;2. GloVe (24 pts)

### TODO 4: Co-occurrence matrix and probability ratios (8 points)

Count $X_{ij}$ on `tweets` (no stop words) with a symmetric window of **2** and plain counts (every neighbor adds 1, no
distance weighting). Then compute, with the lecture's notation,

$$P_{ik} = P(k \mid i) = \frac{X_{ik}}{X_i}, \qquad X_i = \sum_k X_{ik}$$

Careful: $X_i$ is the **row sum of the co-occurrence matrix**, which is *not* the number of times word $i$ occurs.

Print the lecture's table for **your** pair: $i$, $j$ and the four probe words $k$ of `MY['pair']`.
(Every probe word co-occurs with both $i$ and $j$, so no denominator is zero.)

**Check:** `X['nvidia']['rtx'] = 21`. The probabilities and ratios are not shown.

In [ ]:
# TODO 4 (8 pts): X_ij, X_i and P(k | i)

X = defaultdict(Counter)
for t in tweets:
    for i, w in enumerate(t):
        for j in range(max(0, i - 2), min(len(t), i + 3)):
            if j != i:
                ________

print("X['nvidia']['rtx'] =", X['nvidia']['rtx'])


def P(k, i):
    X_i = ________
    return ________


wi, wj, probes = MY['pair']
print(f"{'k':<12}{'P(k|' + wi + ')':>22}{'P(k|' + wj + ')':>22}{'ratio':>9}")
for k in probes:
    ratio = ________
    print(f"{k:<12}{P(k, wi):>22.5f}{P(k, wj):>22.5f}{ratio:>9.3f}")

### TODO 5: The weighting function (4 points)

$$f(x) = \begin{cases} (x / x_{max})^{\alpha} & x < x_{max} \\ 1 & \text{otherwise} \end{cases}$$

Write it **vectorized** (no Python loop), so that it accepts a whole array. Our corpus is tiny compared with GloVe's,
so this assignment uses $x_{max} = 10$ and $\alpha = 3/4$. Then report, over all non-zero cells of your `X`, how many
receive the full weight $f = 1$ and the mean weight.

**Check:** `glove_weight([0, 1, 5, 10, 50]) -> [0. 0.178 0.595 1. 1.]`.

In [ ]:
# TODO 5 (4 pts): vectorized f(x)

def glove_weight(x, x_max=10, alpha=0.75):
    x = np.asarray(x, dtype=float)
    return ________


print('glove_weight([0, 1, 5, 10, 50]) ->', glove_weight([0, 1, 5, 10, 50]).round(3))

values = np.array([v for row in X.values() for v in row.values()])
weights = glove_weight(values)
print('#non-zero cells     :', len(values))
print('#cells with f = 1   :', ________)
print(f'mean weight         : {________:.4f}')

### TODO 6: Analogies by hand (12 points)

`a : b = c : ?` means $b - a + c$. With unit-length vectors (`Vn`, given; restricted to the 100,000 most frequent
words for speed) the answer is the word whose vector has the **largest dot product** with $q = \hat{b} - \hat{a} + \hat{c}$.

1. Complete `analogy(a, b, c, exclude)`. With `exclude=True` the three query words can **not** be the answer;
   with `exclude=False` **every** word of the vocabulary can.
2. Complete the gensim call so that it asks the **same** question `a : b = c : ?`.
3. Evaluate **your** four sections `MY['sections']` (the first 300 usable questions of each) with three methods: gensim,
   by hand with exclusion, by hand without exclusion, and count how often the no-exclusion answer is one of the three query words.

**Check:** `analogy('man', 'king', 'woman', exclude=True) -> queen`, and in every row of your table
`gensim` and `hand (exclude)` are **identical**. Nothing else is shown.

In [ ]:
ANALOGY_URL = "https://raw.githubusercontent.com/snhzyn/2026-2-Unstructured-Data-Analysis/main/week06-text-representation-2/data/word_analogy_goldstandard.txt"

sections, current = {}, None
for line in pd.read_csv(ANALOGY_URL, header=None, sep='\t')[0]:
    line = line.strip()
    if line.startswith(':'):
        current = line[2:]
        sections[current] = []
    elif len(line.split()) == 4:
        sections[current].append(line.lower().split())

TOP = 100_000
Vn = glove.vectors[:TOP] / np.linalg.norm(glove.vectors[:TOP], axis=1, keepdims=True)   # given
print({name: len(q) for name, q in sections.items()})

In [ ]:
# TODO 6 (12 pts): a : b = c : ?

def analogy(a, b, c, exclude=True):
    q = ________
    scores = ________
    if exclude:
        scores[________] = -np.inf
    return glove.index_to_key[int(np.argmax(scores))]


print("analogy('man', 'king', 'woman', exclude=True)  ->", analogy('man', 'king', 'woman', exclude=True))
print("analogy('man', 'king', 'woman', exclude=False) ->", analogy('man', 'king', 'woman', exclude=False))

rows = []
for name in MY['sections']:
    qs = [q for q in sections[name] if all(w in k2i and k2i[w] < TOP for w in q)][:300]
    g_ok = ex_ok = no_ok = no_input = 0
    for a, b, c, d in qs:
        g = glove.most_similar(positive=________, negative=________, topn=1, restrict_vocab=TOP)[0][0]
        ex = analogy(a, b, c, exclude=True)
        no = analogy(a, b, c, exclude=False)
        g_ok += (g == d)
        ex_ok += (ex == d)
        no_ok += (no == d)
        no_input += ________
    rows.append({'section': name, 'n': len(qs), 'gensim': g_ok / len(qs), 'hand (exclude)': ex_ok / len(qs),
                 'hand (no exclude)': no_ok / len(qs), 'no-exclude returns a query word': no_input})

table = pd.DataFrame(rows).set_index('section')
print(table.round(3))

## &nbsp;3. FastText (12 pts)

### TODO 7: Character n-grams (6 points)

Follow the lecture's *Subword model* slide literally:

* add `<` at the beginning and `>` at the end of the word,
* keep **all** n-grams with $n$ = `n_min` … `n_max` (3 to 6), from the shortest $n$ to the longest, left to right,
* **include the word itself** (`<word>` with the markers) in the set, but do not add it twice when it is already one of the n-grams.

**Check:** `her` → `6` items, `['<he', 'her', 'er>', '<her', 'her>', '<her>']`. Your two words `MY['ngram_words']` are not shown.

In [ ]:
# TODO 7 (6 pts): the subword set of a word

def char_ngrams(word, n_min=3, n_max=6):
    w = ________
    grams = [________ for n in range(n_min, n_max + 1) for i in range(________)]
    if ________:
        grams.append(w)
    return grams


for word in ['her'] + MY['ngram_words']:
    g = char_ngrams(word)
    print(f'{word:<12} {len(g):>3} items  first: {g[:3]}  last: {g[-2:]}')

### TODO 8: A FastText vector, rebuilt (6 points)

The model below is given. For any word, `ft_ngram_hashes` returns the rows of `ft.wv.vectors_ngrams` that hold its
n-gram vectors (n = 3…6, *without* the whole word).

* (a) your word `MY['oov']` is **not** in the vocabulary. Build its vector from the n-gram vectors, once as the **sum**
  (the lecture) and once as the **mean**, and check which one gensim returns.
* (b) your word `MY['inv']` **is** in the vocabulary, so it also has a vector of its own, `ft.wv.vectors_vocab[idx]`
  (the whole word is part of its set, TODO 7). gensim returns the **mean over the n-gram vectors and the word's own vector**.
  Write that expression.

Print the four booleans. **Check:** for both of your words (all have 5+ letters) the number of n-grams gensim uses equals
`len(char_ngrams(word)) - 1` from TODO 7: the whole word is the one item gensim keeps apart. The booleans are not shown.

In [ ]:
ft = FastText(sentences=tweets_full, vector_size=50, window=2, min_count=3,
              sg=1, workers=1, seed=42, epochs=10, min_n=3, max_n=6)
print('vocabulary:', len(ft.wv), ' n-gram buckets:', ft.wv.vectors_ngrams.shape)

In [ ]:
# TODO 8 (6 pts): sum, mean, and the in-vocabulary case

# (a) an out-of-vocabulary word  (atol=1e-6 absorbs float32 rounding)
word = MY['oov']
hashes = ft_ngram_hashes(word, ft.wv.min_n, ft.wv.max_n, ft.wv.bucket)
ngram_vecs = ________
v_sum = ________
v_mean = ________
print(f'{word}: in vocab={word in ft.wv.key_to_index}  #n-grams: {len(hashes)}  len(char_ngrams)-1: {len(char_ngrams(word)) - 1}')
print('   sum  == gensim:', np.allclose(v_sum, ft.wv[word], atol=1e-6))
print('   mean == gensim:', np.allclose(v_mean, ft.wv[word], atol=1e-6))

# (b) an in-vocabulary word
word = MY['inv']
idx = ft.wv.key_to_index[word]
hashes = ft_ngram_hashes(word, ft.wv.min_n, ft.wv.max_n, ft.wv.bucket)
ngram_vecs = ft.wv.vectors_ngrams[hashes]
v_ngram_mean = ngram_vecs.mean(axis=0)
v_full = ________
print(f'{word}: in vocab={word in ft.wv.key_to_index}  #n-grams: {len(hashes)}  len(char_ngrams)-1: {len(char_ngrams(word)) - 1}')
print('   n-gram mean only        == gensim:', np.allclose(v_ngram_mean, ft.wv[word], atol=1e-6))
print('   with its own word vector == gensim:', np.allclose(v_full, ft.wv[word], atol=1e-6))

## &nbsp;4. Document Level (20 pts)

### TODO 9: Averaged GloVe vectors as a document vector (8 points)

A tweet's vector is the **mean of the GloVe vectors of its tokens** (`tweets`, no stop words):

* a token that occurs twice counts **twice** (do not deduplicate),
* tokens missing from GloVe are skipped,
* a tweet with no token in GloVe gets the **zero vector** of length 50 (not `NaN`, not removed from the matrix).

Then search the 5 nearest tweets of **your** query tweet `MY['query']` by cosine similarity (the query itself excluded),
and compute the share of neighbors with the **same** sentiment label as the query.

**Check:** `D.shape = (10282, 50)`. Nothing else is shown.

In [ ]:
# TODO 9 (8 pts): mean word vector per tweet, then cosine search

def doc_vector(tokens):
    vecs = [________ for w in tokens if ________]
    return ________


D = np.vstack([doc_vector(t) for t in tweets])
print('D.shape =', D.shape)
print('#tweets with a zero vector:', ________)

QUERY_ID = MY['query']
sim = cosine_similarity(________, D)[0]
order = [i for i in np.argsort(-sim) if i != QUERY_ID][:5]

print('\nQUERY :', df['Review'][QUERY_ID][:100])
for rank, i in enumerate(order, 1):
    print(f'{rank}. cos={sim[i]:.3f}  [{df["Sentiment"][i]:8}] {df["Review"][i][:80]}')

same_label = ________
print(f'\nneighbors with the query label: {round(same_label * 5)} of 5 (share {same_label:.1f})')

### TODO 10: Doc2Vec, PV-DM vs PV-DBOW (12 points)

Train both Paragraph Vector models of the lecture on `tweets`, with the paragraph id = the row number of the tweet.

* Fill in `dm` so that the loop really trains **PV-DM** first and **PV-DBOW** second.
* Before training, keep a **snapshot** of the word matrix `model.wv.vectors`, and after training check whether the
  word vectors changed. The snapshot must be an independent copy of the numbers: if `before` and `model.wv.vectors`
  end up being the same array in memory, the check is meaningless.
* Infer a vector for **your** new tweet `MY['new_tweet']` with the PV-DBOW model and list its 3 nearest training tweets.

Each model trains in about 10 seconds. **Check:** `paragraph matrix: (10282, 50)` for both. The booleans are not shown.

In [ ]:
# TODO 10 (12 pts): PV-DM and PV-DBOW, and what each one trains

tagged = [TaggedDocument(words=t, tags=________) for i, t in enumerate(tweets)]

models = {}
for name, dm in [('PV-DM', ________), ('PV-DBOW', ________)]:
    model = Doc2Vec(dm=dm, dbow_words=0, vector_size=50, window=2, min_count=3,
                    epochs=20, workers=1, seed=42)
    model.build_vocab(tagged)
    before = ________
    model.train(tagged, total_examples=________, epochs=model.epochs)
    unchanged = ________
    models[name] = model
    print(f'{name:<8} paragraph matrix: {model.dv.vectors.shape}   word vectors unchanged: {unchanged}')

new_tweet = MY['new_tweet']
v_new = models['PV-DBOW'].infer_vector(new_tweet, epochs=50)
print('\nnearest tweets to', new_tweet)
for doc_id, score in models['PV-DBOW'].dv.most_similar(________, topn=3):
    print(f'   {score:.3f}  {df["Review"][doc_id][:80]}')

## &nbsp;5. Reading Your Own Results (20 pts)

Five questions, 4 points each. **(A)** is a value you read from **your own** outputs above (2 points), **(B)** is a
multiple-choice item about why that output looks the way it does (2 points). Each has exactly one correct answer.

* Type every answer as a **literal**: a string in quotes (`'love'`, `'C'`) or an integer (`3`).
  An answer written as code (for example `table['gensim'].idxmin()`) gets **0** for that item, even if it evaluates correctly.
* (A) answers are compared with the values the grader computes for **your** `STUDENT_ID`.

**Q1 (TODO 2).**
(A) Among your three words `MY['words']`, which one has the **highest** removal probability at $t = 10^{-3}$?
(B) At $t = 10^{-3}$ some words get $P(w) = 0$. Why?

> A. Their relative frequency is below $t$, so $1 - \sqrt{t / f(w)}$ is negative and is clipped to 0.
> B. gensim's `min_count` removes them before subsampling.
> C. They are stop words and were removed in Section 0.
> D. $t = 10^{-3}$ is larger than 1 for those words.

**Q2 (TODO 3).**
(A) What is the most probable **context** word of your word `MY['center']`?
(B) If `w2v.syn1neg` is replaced by `w2v.wv.vectors` in `context_distribution`, the most probable "context" word is,
in most cases, the center word itself. Why?

> A. `syn1neg` is normalized to unit length and `wv.vectors` is not.
> B. Negative sampling stores the vocabulary in reverse order.
> C. The dot product of $v_c$ with itself, $\lVert v_c \rVert^2$, is usually larger than its dot product with any other row of the same matrix.
> D. The softmax always gives the highest probability to the first word of the vocabulary.

**Q3 (TODO 4).**
(A) Among your four probe words, which one has the ratio $P(k \mid i) / P(k \mid j)$ **closest to 1**?
(B) Following the lecture's ice / steam table, a ratio close to 1 means that the probe word $k$ ...

> A. never co-occurs with $i$ or with $j$.
> B. is about equally related to $i$ and $j$: related to both, or to neither.
> C. is a synonym of $i$.
> D. has $X_k = 0$.

**Q4 (TODO 6).**
(A) Which of your four sections has the **lowest gensim accuracy** in your table? (type the section name)
(B) Without exclusion, the by-hand answer to $\hat{b} - \hat{a} + \hat{c}$ is very often one of the query words. Why?

> A. `restrict_vocab=100000` removes the correct answer.
> B. The vectors in `Vn` are not normalized.
> C. The analogy file contains mistakes.
> D. The offset $\hat{b} - \hat{a}$ is small compared with the vectors themselves, so $q$ stays closest to $\hat{c}$ (or $\hat{b}$).

**Q5 (TODO 9, 10).**
(A) How many of the 5 neighbors of your query tweet in TODO 9 carry the **same** sentiment label as the query? (an integer 0–5)
(B) In TODO 10, why are the word vectors of PV-DBOW (`dbow_words=0`) unchanged after training?

> A. PV-DBOW predicts words from the paragraph vector alone; the word vectors are never used, so no gradient reaches them.
> B. `.copy()` freezes the word matrix.
> C. 20 epochs are too few to change them.
> D. `min_count=3` removes every word.

In [ ]:
# Part 5 (20 pts): literals only - a string in quotes or an integer

Q1A = ________
Q1B = ________

Q2A = ________
Q2B = ________

Q3A = ________
Q3B = ________

Q4A = ________
Q4B = ________

Q5A = ________
Q5B = ________

print('STUDENT_ID:', STUDENT_ID)
for name in ['Q1A', 'Q1B', 'Q2A', 'Q2B', 'Q3A', 'Q3B', 'Q4A', 'Q4B', 'Q5A', 'Q5B']:
    print(f'{name}: {globals()[name]!r}')

---
### ✅ Before you submit

- [ ] `Runtime ▸ Restart and run all` finishes with no error
- [ ] Every `________` is filled in and every output cell is visible
- [ ] Every **Check** line of the assignment matches your output
- [ ] `STUDENT_ID` is **your own**, and every Part 5 answer is a typed literal read from **your own** outputs
- [ ] The file is named `Assignment_3_YourName_StudentID.ipynb`